# Central New Zealand fault-mesh pipeline

This notebook builds triangulated fault-surface meshes for the central New
Zealand Community Fault Model, from raw fault traces all the way to clean,
uniformly-sized triangle meshes. It combines what used to be three separate
scripts into a single, ordered workflow:

| Stage | What it does | Outputs |
|-------|--------------|---------|
| **1. Build surfaces** | Turn each (multi-segment) fault into a 3-D triangulated surface from depth contours | `test_objs/`, `test_vtks/`, `test_contours/` |
| **2. Cut surfaces** | Trim each surface where higher-priority faults cross it, and against the base depth surface | `test_final_meshes/*_cut.obj` |
| **3. Remesh** | Optimise every cut surface into uniform, near-equilateral triangles with **MMG** | `test_remeshed_vtks/`, `merged_mesh_remeshed.vtk` |

**Run the cells in order, top to bottom.** The stages are sequential: each one
consumes the files written by the previous one, so a single linear run produces
everything. You can also re-run a later stage on its own (e.g. re-cut without
re-building) because each stage reads its inputs back from disk.

> **Prerequisites**: run this notebook from its own folder
> (`scratch/central_nz_no_leapfrog/`) inside the `leapfrog-fault-models` conda
> environment. All paths below are relative to that folder.


## Setup

### Imports

In [ ]:
import os
from pathlib import Path

import numpy as np
import meshio
import pyvista as pv

from fault_mesh.faults.leapfrog import LeapfrogMultiFault
from fault_mesh.faults.mesh import FaultMesh
from fault_mesh.io.array_operations import read_raster

### Configuration

Every input path and tunable parameter lives in this one cell, so a collaborator
can adapt the pipeline to a new dataset by editing here and nowhere else.

The `*_edited.csv` files are **human-curated** inputs (fault groupings, cutting
order, trace extensions) — they are reviewed and edited by hand, not generated
automatically. See the comments by each path.

In [ ]:
# --- Input files (relative to this notebook's folder) ---
DOCS = Path("../../docs/tutorials")
FAULT_SHP         = DOCS / "tutorial_gis/central_nz_minimal_data.shp"           # raw fault traces
FAULT_SYSTEMS     = DOCS / "define_connections_data/central_gt1_5_connected_edited.csv"  # which segments form one fault
CUTTING_HIERARCHY = DOCS / "define_connections_data/central_gt1_5_hierarchy_edited.csv"  # which fault cuts which
DEPTH_RASTER      = Path("with_hannu_mods.tif")          # base/Moho-style depth surface (GeoTIFF, z in metres)
ADDITIONAL_CUTS   = Path("additional_cuts.csv")          # extra cut relationships not implied by the hierarchy
# EXCLUDED_CUTS   = Path("excluded_cuts.csv")            # OPTIONAL: pairs that must NEVER be cut (not used here)
TRACE_EXT_EDITED  = Path("trace_extensions_edited.csv")  # curated along-strike trace extensions

# --- Coordinate system & network-building parameters ---
EPSG             = 2193      # NZTM2000; set to None to skip reprojection
DIST_TOLERANCE   = 1000.0    # max gap (m) for two trace segments to be treated as connected

# --- Stage 1: surface meshing ---
DEPTH_CONTOUR_LEVELS = np.arange(0., 32000., 500.)  # depths (m) at which to draw contours
MESH_RESOLUTION      = 500.0                         # target triangle size (m) for the raw surface

# --- Stage 2: cutting ---
MIN_CUT_DISTANCE = 2.0e3     # don't cut faults whose meshes are closer than this (m)
BOTTOM_DEPTH     = -31500.0  # depth (m) used when reasoning about cut extents

# --- Stage 3: MMG remeshing ---
TARGET_SIZE     = 1000.0     # uniform target edge length (m)
HAUSD           = 500.0      # max allowed deviation of the remesh from the original surface (m)
RIDGE_ANGLE_DEG = 45.0       # dihedral angle above which kinks are preserved

# --- Output directories ---
OBJ_DIR     = Path("test_objs")           # raw surfaces (OBJ) -- input to Stage 2
VTK_DIR     = Path("test_vtks")           # raw surfaces (VTK) -- for inspection
CONTOUR_DIR = Path("test_contours")       # depth contours (GeoJSON)
FINAL_DIR   = Path("test_final_meshes")   # cut surfaces -- input to Stage 3
REMESH_DIR  = Path("test_remeshed_vtks")  # remeshed surfaces
MERGED_VTK  = Path("merged_mesh_remeshed.vtk")

for d in (OBJ_DIR, VTK_DIR, CONTOUR_DIR, FINAL_DIR, REMESH_DIR):
    d.mkdir(exist_ok=True)

### Build the fault network

This is the shared foundation for every stage. We:

1. read the raw fault traces from the shapefile and reproject to NZTM;
2. **find connections** — segments whose ends are within `DIST_TOLERANCE` are
   candidates to belong to the same fault;
3. **read fault systems** — the curated CSV says which segments actually form
   one continuous (multi-segment) fault, and we generate those *curated faults*;
4. **read the cutting hierarchy** — the curated CSV that ranks faults so we know,
   later, which fault truncates which.

`fault_data` is built **once** here and reused by all three stages.

In [ ]:
fault_data = LeapfrogMultiFault.from_nz_cfm_shp(
    str(FAULT_SHP), remove_colons=True, epsg=EPSG,
    smoothing_n=None, dip_multiplier=1.0, exclude_zero=False)
fault_data.segment_distance_tolerance = DIST_TOLERANCE
fault_data.find_connections(verbose=False)

# Curated multi-segment faults, and the order in which faults cut each other.
fault_data.read_fault_systems(str(FAULT_SYSTEMS))
fault_data.generate_curated_faults()
fault_data.read_cutting_hierarchy(str(CUTTING_HIERARCHY))

print(f"{len(fault_data.curated_faults)} curated faults; "
      f"cutting hierarchy has {len(fault_data.cutting_hierarchy)} entries")

### Trace extensions (human-in-the-loop)

Faults often need to be extended a little along strike so that neighbouring
faults actually meet and cut cleanly. This is a **two-step, human-reviewed**
process:

1. `suggest_trace_extensions(...)` writes a CSV (and GeoJSON) of *proposed*
   extensions for you to inspect in GIS.
2. You edit/approve them, save as `trace_extensions_edited.csv`, and
   `read_trace_extensions(...)` + `apply_trace_extensions()` apply the curated
   result.

The edited file already exists, so the cell below runs end-to-end. Re-running the
`suggest` step simply regenerates the suggestions; it does **not** overwrite your
edited file.

> Note: we apply the extensions to the single shared `fault_data` before meshing,
> so the surfaces built in Stage 1 already reflect them.

In [ ]:
# Step 1: propose extensions (for review in GIS). Safe to re-run.
fault_data.suggest_trace_extensions(
    out_file="suggested_trace_extensions.csv",
    geojson_out_file="suggested_trace_extensions.geojson",
    fit_distance=5.e3, extend_distance=40.e3, proximity_threshold=1.e3)

# Step 2: apply the curated, hand-edited extensions.
fault_data.read_trace_extensions(str(TRACE_EXT_EDITED))
fault_data.apply_trace_extensions()

## Stage 1 — Build fault surfaces

For each curated fault we draw a stack of **depth contours** (lines of equal
depth, every 500 m down to 32 km), then triangulate the surface spanning those
contours. The result is a raw 3-D fault surface.

If the proper surface-meshing step fails for a fault (it can, for awkward
geometries), we fall back to a simpler contour-based mesh and save the offending
contours to a `failed_contours_*.geojson` for inspection — so one bad fault never
stops the whole run.

Each surface is written three ways: **OBJ** (input to Stage 2), **VTK** (for
quick viewing), and the **contours as GeoJSON**.

In [ ]:
for fault in fault_data.curated_faults:
    print(fault.name)
    try:
        fault.generate_depth_contours(DEPTH_CONTOUR_LEVELS, smoothing=False)
        mesh = fault.mesh_fault_surface(check_mesh=False, resolution=MESH_RESOLUTION)
    except Exception as e:
        # Fallback: simple contour mesh, and save the contours that tripped us up.
        print(f"  Failed to mesh {fault.name}: {e}")
        fault.contours.to_file(f"failed_contours_{fault.name}.geojson", driver="GeoJSON")
        mesh = fault.mesh_simple_contours(DEPTH_CONTOUR_LEVELS)

    fault.contours.to_file(str(CONTOUR_DIR / f"{fault.name}_depth_contours.geojson"), driver="GeoJSON")
    mesh.write(str(VTK_DIR / f"{fault.name}_depth_contours.vtk"))
    mesh.write(str(OBJ_DIR / f"{fault.name}_depth_contours.obj"))

## Stage 2 — Cut the surfaces

Now we truncate faults against each other and against the base depth surface, so
the final meshes don't overlap or extend below the model.

### Stage 2 setup

We load:

- the **depth raster** as a PyVista surface (the lower bound everything is
  trimmed to), and
- the **additional cuts** CSV (extra truncation relationships not captured by the
  hierarchy alone).

There is also an optional inverse, `read_excluded_cuts(...)`, for pairs that must
**never** be cut. The cutting step consults both override lists via
`fault_data.should_cut(...)`: `additional_cuts` is loaded here, while
`excluded_cuts` is left as a commented example (no such pairs in this system).
Both default to empty sets, so to use exclusions elsewhere just uncomment the
`EXCLUDED_CUTS` path and its loader.

Then we read the Stage 1 OBJ surfaces back from disk into each fault's `.mesh`.
Reading from disk (rather than reusing in-memory objects) is deliberate: it keeps
this stage self-contained, so you can re-cut without re-running Stage 1.

In [ ]:
depth_pyvista = read_raster(str(DEPTH_RASTER), use_z=True, out_crs=f"EPSG:{EPSG}")
fault_data.read_additional_cuts(str(ADDITIONAL_CUTS))
# OPTIONAL override list of pairs that must NEVER be cut. Same 2-column,
# header-less CSV format as additional_cuts: each row is `fault_to_cut,cutting_fault`.
# Not needed for this fault system, but uncomment to use it elsewhere:
# fault_data.read_excluded_cuts(str(EXCLUDED_CUTS))

# Load the Stage 1 surfaces back onto each fault.
for fault in fault_data.curated_faults:
    obj_file = OBJ_DIR / f"{fault.name}_depth_contours.obj"
    if obj_file.exists():
        fault.mesh = FaultMesh.from_file(str(obj_file))

# Lookup of faults that actually have a mesh, used during cutting.
cutting_dict = {f.name: f for f in fault_data.curated_faults if f.mesh is not None}
print(f"{len(cutting_dict)} faults have meshes available for cutting")

### Apply the cutting hierarchy

We walk the faults in hierarchy order. For each fault we consider every
**higher-priority** fault. The manual override lists are checked first via
`should_cut(...)` (a pair in `additional_cuts` is always cut, one in
`excluded_cuts` never); otherwise we fall back to `decide_whether_to_cut(...)`,
which asks whether they genuinely intersect and are far enough apart to warrant a
cut (the `higher_meshes` give the context of faults that already truncate the
cutter). If a cut is called for, we build a *cutting fragment* from the cutter and
slice this fault with it, keeping the side nearest the fault's own trace.

Finally each fault is trimmed against the depth surface with `cut_mesh_pv(...)`
and the result written as `<name>_cut.obj` — the input to Stage 3.

`fancy_cutting=True` uses the more robust intersection-following cut.

In [ ]:
threshold = 0.7  # currently unused by the library; kept for API compatibility

for fault_name in fault_data.cutting_hierarchy:
    print(f"Processing {fault_name} ...")
    fault = fault_data.name_dict[fault_name]
    if fault.mesh is None:
        continue

    # Faults ranked above this one are candidates to cut it.
    faults_to_cut = fault_data.cutting_hierarchy[:fault_data.cutting_hierarchy.index(fault_name)]
    for cut_name in faults_to_cut:
        if cut_name not in cutting_dict:
            continue
        higher = fault_data.cutting_hierarchy[:fault_data.cutting_hierarchy.index(cut_name)]
        higher_meshes = [cutting_dict[n].mesh for n in higher]
        cutting_mesh = cutting_dict[cut_name].mesh
        # Manual overrides win first. should_cut() matches on the curated fault
        # names (reliable) and returns True/False to force/skip a cut, or None to
        # fall through to the geometric test below. (We match on names here rather
        # than via decide_whether_to_cut's excluded_cuts/additional_cuts kwargs,
        # because mesh names carry a "_depth_contours"/"_cut_by_" suffix that would
        # not match the bare names in the CSVs.) additional_cuts/excluded_cuts
        # default to empty sets, so this is a no-op unless you loaded the CSVs.
        decision = fault_data.should_cut(fault_name, cut_name)
        if decision is None:
            decision = fault.mesh.decide_whether_to_cut(cutting_mesh, threshold=threshold,
                                                        min_distance=MIN_CUT_DISTANCE,
                                                        higher_meshes=higher_meshes,
                                                        bottom_depth=BOTTOM_DEPTH, fancy_cutting=True)
        if decision:
            print(f"  Cutting {fault.name} by {cut_name}")
            fragment = cutting_mesh.generate_cutting_mesh(fault.mesh, max_distance=5.e3)
            new_mesh = fault.mesh.cut_mesh(cutting_mesh,
                                           fault_trace=fault.original_nztm_trace_array,
                                           cutting_fragment=fragment, fancy_cutting=True)
            fault_data.name_dict[fault.name].mesh = new_mesh
            cutting_dict[fault.name].mesh = new_mesh

    # Trim against the base depth surface, then save the final cut mesh.
    try:
        depth_trimmed = fault_data.name_dict[fault.name].mesh.cut_mesh_pv(
            depth_pyvista, fault_trace=fault.original_nztm_trace_array)
        fault_data.name_dict[fault.name].mesh = depth_trimmed
    except Exception as e:
        print(f"  Error trimming depth for {fault.name}: {e}")
        continue
    fault_data.name_dict[fault.name].mesh.mesh.write(str(FINAL_DIR / f"{fault.name}_cut.obj"))
    print(f"  Done: {fault.name}")

## Stage 3 — Remesh with MMG

The cut surfaces have uneven, often sliver-shaped triangles left over from the
contouring and cutting. This stage cleans each one into **uniform, near-
equilateral triangles** of `TARGET_SIZE` using **MMG** (`mmgs`).

### Why MMG and not a parametrise-and-regenerate remesher?

A reparametrising remesher (e.g. gmsh `classifySurfaces` + `createGeometry`)
flattens the whole surface onto a 2-D plane and re-triangulates there. For a
long, **warped** fault sheet like the Alpine Fault, that flattening *folds over
itself*, and the folded region is silently dropped — **about half of the Alpine
Fault was being deleted.**

MMG instead remeshes the surface **in place**, using local edge
split/collapse/swap and moving nodes back onto the original surface. There is no
global flattening to fold, so:

- nothing is cut off — the fault outline and full extent are preserved
  (area preserved to >99.8 % across all faults), and
- triangle quality jumps from a median of ~0.74 to ~0.97 (1.0 = equilateral).

`HAUSD` controls how tightly the new mesh hugs the original surface; `RIDGE_ANGLE_DEG`
tells MMG which sharp kinks to preserve rather than smooth.

In [ ]:
remeshed_paths = []
for obj_path in sorted(FINAL_DIR.glob("*_cut.obj")):
    name = obj_path.stem.removesuffix("_cut")
    out_path = REMESH_DIR / f"{name}_remeshed.vtk"
    print(f"Remeshing {name}")
    try:
        fault_mesh = FaultMesh.from_file(str(obj_path))
        remeshed = fault_mesh.remesh(target_size=TARGET_SIZE, hausd=HAUSD,
                                     ridge_angle_deg=RIDGE_ANGLE_DEG, verbose=False)
        meshio.write(str(out_path), remeshed.mesh)
        remeshed_paths.append(out_path)
    except Exception as e:
        print(f"  failed: {e}")

### Merge for inspection

Finally we merge every remeshed surface into one VTK so the whole fault network
can be opened and checked in one go (e.g. in ParaView).

In [ ]:
meshes = [pv.from_meshio(meshio.read(str(p))) for p in remeshed_paths]
combined = pv.merge(meshes)
combined.save(str(MERGED_VTK))
print(f"Merged {len(meshes)} remeshed surfaces -> {MERGED_VTK}")

### (Optional) Quick 3-D view

A quick look at the merged result. This needs an interactive/display-capable
environment; if it doesn't render inline, set a PyVista Jupyter backend
(e.g. `pv.set_jupyter_backend("trame")`) or open `merged_mesh_remeshed.vtk` in
ParaView instead.

In [ ]:
combined = pv.read(str(MERGED_VTK))
plotter = pv.Plotter()
plotter.add_mesh(combined, show_edges=True, color="lightgray")
plotter.show()